In [1]:
import boto3

In [3]:
session = boto3.Session(
    region_name='us-east-1',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

endpoint = 'http://localhost:4566'

In [4]:
kinesis = session.client('kinesis', endpoint_url=endpoint)

In [17]:
shards = kinesis.list_shards(StreamName='pr-events-stream')

In [22]:
shards['Shards']

[{'ShardId': 'shardId-000000000000',
  'HashKeyRange': {'StartingHashKey': '0',
   'EndingHashKey': '340282366920938463463374607431768211455'},
  'SequenceNumberRange': {'StartingSequenceNumber': '0',
   'EndingSequenceNumber': '1795296782364'}},
 {'ShardId': 'shardId-000000000001',
  'ParentShardId': 'shardId-000000000000',
  'HashKeyRange': {'StartingHashKey': '0',
   'EndingHashKey': '170141183460469231731687303715884105727'},
  'SequenceNumberRange': {'StartingSequenceNumber': '1795296782364'}},
 {'ShardId': 'shardId-000000000002',
  'ParentShardId': 'shardId-000000000000',
  'HashKeyRange': {'StartingHashKey': '170141183460469231731687303715884105728',
   'EndingHashKey': '340282366920938463463374607431768211455'},
  'SequenceNumberRange': {'StartingSequenceNumber': '1795296782364'}}]

In [39]:
last_sequence_number = {}
for shard in shards['Shards']:
    shard_iterator = kinesis.get_shard_iterator(StreamName='pr-events-stream', ShardId=shard['ShardId'],ShardIteratorType='TRIM_HORIZON')
    records = kinesis.get_records(ShardIterator=shard_iterator['ShardIterator'], Limit=25)
    record_details = records.get('Records')
    for i in record_details:
        print(i['SequenceNumber'], i['PartitionKey'])
    if len(record_details) > 0:
        last_sequence_number[shard['ShardId']] = i['SequenceNumber']

1795296782367 org/repo-b
1795296782368 org/repo-c
1795296782371 human
1795296782374 human
1795296782376 human
1795296782365 org/repo-a
1795296782366 org/repo-a
1795296782369 org/repo-a
1795296782370 org/repo-d
1795296782372 agent
1795296782373 agent
1795296782375 agent


In [40]:
last_sequence_number

{'shardId-000000000001': '1795296782376',
 'shardId-000000000002': '1795296782375'}

In [41]:
shard_iterator

{'ShardIterator': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMnxUUklNX0hPUklaT058fDB8',
 'ResponseMetadata': {'RequestId': '03a02c87-1b41-4ea4-8a17-5da6d8917c31',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.1;charset=UTF-8',
   'content-length': '92',
   'date': 'Sun, 22 Nov 2026 02:40:54 GMT',
   'x-amz-id-2': '03a02c87-1b41-4ea4-8a17-5da6d8917c31',
   'x-amz-request-id': '03a02c87-1b41-4ea4-8a17-5da6d8917c31',
   'x-amzn-requestid': '03a02c87-1b41-4ea4-8a17-5da6d8917c31'},
  'RetryAttempts': 0}}